# Case-02: RotSpring2D + BeamNL 組合最小測試

**專案**: pyfem-plastic-hinge（延續 Case-01,同一套環境偵測 + 驗證格式）

**前置**: Case-01 已通過（RotSpring2D 孤立線性行為,對手算 M/k 機器精度一致）。
這個 notebook 只在 Case-01 通過的前提下才有意義。

**目標（Stage 3）**：把 `RotSpring2D` 跟 pyFEM 既有的 `BeamNL` 元素串接,
驗證組合後的總撓度是否等於「彈簧貢獻 + 梁本身撓度」的手算疊加值。
對應 calculix-hinge2 專案的 UB+HINGE2 Stage 4 combination test。

**模型**：節點A(地面,完全固定) —`RotSpring2D`(只提供轉角勁度k)— 節點B
(平移直接用邊界條件鎖死,不透過彈簧) —`BeamNL`— 節點C(自由端,施力)。

**手算**（疊加法,結構在轉角彈簧處靜定,反力彎矩 M0=P·L 與彈簧勁度無關）：

$$\theta_0=\frac{M_0}{k}=\frac{PL}{k},\qquad \delta_{beam}=\frac{PL^3}{3EI},\qquad \delta_{total}=\theta_0 L+\delta_{beam}$$


In [ ]:
# ===== 0: 安裝 pyFEM（沿用 Case-01 的環境偵測邏輯，已安裝則略過）=====
import os
if os.path.isdir("/content"):
    PYFEM_DIR = "/content/PyFEM"
else:
    PYFEM_DIR = os.path.join(os.getcwd(), "PyFEM")

if not os.path.isdir(PYFEM_DIR):
    !git clone -q https://github.com/jjcremmers/PyFEM.git {PYFEM_DIR}
    %pip install -q -e {PYFEM_DIR} --break-system-packages
else:
    print(f"{PYFEM_DIR} 已存在，略過安裝")


## 1. 寫入 RotSpring2D（跟 Case-01 完全相同，Case-02 不改元素本身）

In [ ]:
rotspring_code = r'''
from pyfem.elements.Element import Element
from numpy import zeros


class RotSpring2D(Element):
    """2-node 2D 轉角彈簧元素 (pyFEM 自訂元素, Stage 1: 線性彈性)"""

    dofTypes = ['u', 'v', 'rz']

    def __init__(self, elnodes, props):
        Element.__init__(self, elnodes, props)
        self.family = "BEAM"

    def getTangentStiffness(self, elemdat):
        k = elemdat.props.k
        theta1 = elemdat.state[2]
        theta2 = elemdat.state[5]
        dtheta = theta2 - theta1
        M = k * dtheta

        elemdat.fint = zeros(6)
        elemdat.fint[2] = -M
        elemdat.fint[5] = M

        elemdat.stiff = zeros((6, 6))
        elemdat.stiff[2, 2] = k
        elemdat.stiff[2, 5] = -k
        elemdat.stiff[5, 2] = -k
        elemdat.stiff[5, 5] = k

    def getInternalForce(self, elemdat):
        self.getTangentStiffness(elemdat)
'''

target = f"{PYFEM_DIR}/pyfem/elements/RotSpring2D.py"
with open(target, "w") as f:
    f.write(rotspring_code)
print(f"已寫入 {target}")


## 2. Case-02 主測試

`BeamNL` 是幾何非線性(co-rotational)元素,即使小變形範圍也照標準
Newton-Raphson 跑（不假設一步線性收斂）。載重刻意選得小（P=10N,
撓度/長度 <0.02%），避免幾何非線性汙染「線性疊加對不對」這個問題本身
——下一格會用診斷掃描證明這個選擇是對的。


In [ ]:
import sys
sys.path.insert(0, PYFEM_DIR)

from pyfem.util.dataStructures import Properties, GlobalData
from pyfem.fem.NodeSet import NodeSet
from pyfem.fem.ElementSet import ElementSet
from pyfem.fem.DofSpace import DofSpace
from pyfem.fem.Assembly import assembleTangentStiffness
from pyfem.models.ModelManager import ModelManager
from numpy import zeros

# ---------- 參數 ----------
E = 2.0e5; A = 1.0e4; I = 1.0e6; G = E / 2.6
L = 2000.0
k = 3.0e8     # 刻意讓彈簧項跟梁項同量級，才是真正測試組合，不是其中一項獨大
P = 10.0

def build_and_solve(P_applied):
    props = Properties()
    props.HingeElem = Properties({'type': 'RotSpring2D', 'k': k})
    props.BeamElem = Properties({'type': 'BeamNL', 'E': E, 'A': A, 'I': I, 'G': G})

    nodes = NodeSet()
    nodes.add(1, [0.0, 0.0])   # A: 地面
    nodes.add(2, [0.0, 0.0])   # B: 跟A重合，經彈簧連接
    nodes.add(3, [L, 0.0])     # C: 自由端

    elements = ElementSet(nodes, props)
    elements.add(1, 'HingeElem', [1, 2])
    elements.add(2, 'BeamElem', [2, 3])

    dofs = DofSpace(elements)
    cons = dofs.createConstrainer()
    for dtype in ['u', 'v', 'rz']:
        cons.addConstraint(dofs.getForType(1, dtype), 0.0, "main")
    for dtype in ['u', 'v']:
        cons.addConstraint(dofs.getForType(2, dtype), 0.0, "main")
    cons.flush()

    globdat = GlobalData(nodes, elements, dofs)
    globdat.models = ModelManager(props, globdat)

    a = globdat.state
    fext = zeros(len(dofs))
    loadDof = dofs.getForType(3, 'v')
    fext[loadDof] = -P_applied

    for _ in range(20):
        K, fint = assembleTangentStiffness(props, globdat)
        r = fext - fint
        if dofs.norm(r) < 1e-6 * max(P_applied, 1.0):
            break
        da = dofs.solve(K, r)
        a[:] += da[:]
    else:
        raise RuntimeError("Newton-Raphson 未收斂")

    K, fint = assembleTangentStiffness(props, globdat)
    residual = dofs.norm(fext - fint)
    tipDof = dofs.getForType(3, 'v')
    return a[tipDof], residual

def hand_calc(P_applied):
    theta0 = (P_applied * L) / k
    delta_beam = P_applied * L**3 / (3 * E * I)
    return -(theta0 * L + delta_beam)

delta_numeric, residual = build_and_solve(P)
delta_hand = hand_calc(P)
rel_err = abs(delta_numeric - delta_hand) / abs(delta_hand)

print("=== Case-02: RotSpring2D + BeamNL 組合最小測試 ===")
print(f"施加載重 P       = {P:.3f} N")
print(f"總撓度(手算疊加) = {delta_hand:.6f} mm")
print(f"總撓度(數值解)   = {delta_numeric:.6f} mm")
print(f"相對誤差         = {rel_err:.3e}")
print(f"殘差(平衡自我檢查) = {residual:.3e}")

assert rel_err < 1e-5, "組合結果與手算疊加不符"
assert residual < 1e-6, "殘差未收斂"
print("\n✅ PASS")


## 3. 診斷：確認殘餘誤差來自幾何非線性，不是 bug

如果直接用較大的載重（例如 P=1000N，撓度/長度~1.3%），相對誤差會是
1.36e-4 量級——這不是隨便可以放行的數字。做法不是調鬆容忍值，是先證明
它從哪裡來：如果誤差隨載重呈平方衰減，就是 `BeamNL` 的幾何非線性
(co-rotational) 標準特徵，不是元素組合寫錯。


In [ ]:
print("P (N)      數值解        手算解        相對誤差")
for scale in [1.0, 0.5, 0.25, 0.125]:
    Pi = 1000.0 * scale
    di, _ = build_and_solve(Pi)
    hi = hand_calc(Pi)
    ei = abs(di - hi) / abs(hi)
    print(f"{Pi:8.2f}   {di:11.6f}   {hi:11.6f}   {ei:.3e}")
print("\n誤差每次隨 P 減半降到約 1/4 → 平方衰減 → 幾何非線性造成，不是 bug")


## 4. 結論與下一步

實際在這個沙盒環境跑過（不是預期結果）：`P=10N` 時相對誤差 `1.363e-08`，
殘差 `2.668e-07`；`P=1000N` 時誤差 `1.362e-04`，且誤差隨載重呈乾淨的平方
衰減（1000→500→250→125N: 1.362e-4 → 3.407e-5 → 8.518e-6 → 2.129e-6，
每次減半降到約1/4）——確認殘餘誤差是 `BeamNL` 幾何非線性造成，元素組合
本身在小變形線性範圍內是正確的。

**Validation Log**

| ID | 主題 | 比對對象 | 結果 |
|---|---|---|---|
| VL-01 | RotSpring2D 孤立線性行為 | 手算 M/k | 機器精度一致 (rel_err=0) |
| VL-02 | RotSpring2D+BeamNL 組合撓度 | 手算疊加(θ₀L+PL³/3EI) | rel_err=1.363e-08 (P=10N)，且誤差隨P²衰減特徵確認來源是幾何非線性 |

**下一步（Case-03）**：把這個組合的幾何從水平換成垂直柱子方向,驗證座標轉換
在 portal frame 實際會用到的方向上也正確——對應 calculix-hinge2 的
「portal-frame-relevant geometry test」。
